# Herramientas · Código C y ejecutables

Llamar a funciones escritas en C desde Python con `cffi` y empaquetar programas como ejecutables con `pyinstaller`.

## 1. Función C `son_primos`

Crear en C una función `son_primos` que, dada una lista de números, devuelva otra de `bool` indicando si cada uno es primo, y llamarla desde Python. Para compilar se usa el compilador de C que incluye el paquete `ziglang` (instalable con `pip`), así no hace falta instalar Visual Studio; la librería compilada se carga con `ffi.dlopen`.

In [1]:
import os
import subprocess
import sys
import cffi

codigo_c = r"""
#include <stdbool.h>
#ifdef _WIN32
#define EXPORTA __declspec(dllexport)
#else
#define EXPORTA
#endif

static bool es_primo(int n) {
    if (n <= 1) return false;
    if (n <= 3) return true;
    if (n % 2 == 0 || n % 3 == 0) return false;
    for (int i = 5; i * i <= n; i += 6)
        if (n % i == 0 || n % (i + 2) == 0) return false;
    return true;
}

EXPORTA void son_primos(const int *numeros, bool *resultado, int n) {
    for (int i = 0; i < n; i++) resultado[i] = es_primo(numeros[i]);
}
"""
with open("primos.c", "w") as f:
    f.write(codigo_c)
libreria = "primos.dll" if sys.platform == "win32" else "libprimos.so"
subprocess.run([sys.executable, "-m", "ziglang", "cc", "-shared", "-O2", "-o", libreria, "primos.c"], check=True)

ffi = cffi.FFI()
ffi.cdef("void son_primos(const int *numeros, bool *resultado, int n);")
lib = ffi.dlopen(os.path.abspath(libreria))

def son_primos(numeros):
    resultado = ffi.new("bool[]", len(numeros))
    lib.son_primos(ffi.new("int[]", numeros), resultado, len(numeros))
    return list(resultado)

numeros = [1, 2, 3, 4, 17, 21, 777, 7919, 1_000_003]
print(dict(zip(numeros, son_primos(numeros))))
# libera la DLL para poder borrarla
ffi.dlclose(lib)

{1: False, 2: True, 3: True, 4: False, 17: True, 21: False, 777: False, 7919: True, 1000003: True}


## 2. Ejecutable con icono

Convertir un programa en un ejecutable que no necesite Python instalado con `pyinstaller` y, para hacerlo más profesional, añadirle un icono. Desde la terminal sería `pyinstaller --onefile --clean --icon icono.ico saludo.py`. Tarda alrededor de un minuto.

In [2]:
from PIL import Image, ImageDraw

with open("saludo.py", "w", encoding="utf-8") as f:
    f.write('name = input("Dime tu nombre: ")\nprint(f"Hola {name}")\n')
icono = Image.new("RGBA", (256, 256), (0, 0, 0, 0))
ImageDraw.Draw(icono).ellipse((16, 16, 240, 240), fill=(55, 118, 171))
icono.save("icono.ico", sizes=[(256, 256), (64, 64), (32, 32), (16, 16)])

subprocess.run([sys.executable, "-m", "PyInstaller", "--onefile", "--clean", "--noconfirm", "--log-level", "ERROR",
                "--icon", "icono.ico", "--distpath", "dist", "--workpath", "build", "saludo.py"], check=True)
ejecutable = os.path.join("dist", "saludo.exe" if sys.platform == "win32" else "saludo")
print(subprocess.run([ejecutable], input="Ana\n", capture_output=True, text=True).stdout)
print(f"{os.path.getsize(ejecutable) / 1e6:.1f} MB")

Dime tu nombre: Hola Ana

7.3 MB
